In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_customer=f"{catalog_name}.bronze.customer"
silver_customer=f"{catalog_name}.silver.customer"
silver_batchdate=f"{catalog_name}.silver.batchdate"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
if batch_id == "1":
    print("Batch 1 detected. Skipping execution for this notebook.")
    dbutils.notebook.exit("Skipped: Notebook only applicable for Batches 2 and 3")

In [0]:
df_bronze=spark.table(bronze_customer).filter(col("_batch")==batch_id)

In [0]:
df_bronze.printSchema()

In [0]:
spark.table(silver_batchdate).printSchema()

In [0]:
batchdate=spark.table(silver_batchdate)\
    .filter(col("batchid") == batch_id)\
    .select("batchdate").collect()
# batchdate

In [0]:
# df_bronze.select("C_TIER").distinct().display()

In [0]:
# df_bronze.select("CDC_FLAG").distinct().display()

In [0]:
# df_bronze.select("C_ST_ID").distinct().display()

In [0]:
#Deduplication
df_bronze.createOrReplaceTempView("df_bronze")
df_dedup=spark.sql("""
                with ranked as (
                    select * ,row_number() over (
                        partition by C_ID
                        order by cast(cdc_dsn as bigint) desc
                    ) as rn
                    from df_bronze
                ) 
                select * except (rn)
                from ranked 
                where rn=1
            """)
df_dedup.count()

In [0]:
# Renamiing column  and type casting and adding column if needed to avoid errors in appending in silver table
df_silver = (
    df_dedup.withColumn("ActionTS", to_timestamp(lit(batchdate[0][0])))
    .withColumn("C_ID", col("C_ID").cast("long"))
    .withColumn("C_TIER", expr("try_cast(C_TIER as TINYINT)"))
    .withColumn("C_DOB", col("C_DOB").cast("date"))
    .withColumnRenamed("C_ST_ID", "Status")
    .withColumnRenamed("C_EMAIL_1", "C_PRIM_EMAIL")
    .withColumnRenamed("C_EMAIL_2", "C_ALT_EMAIL")
)
# hadndling phone for df silver
df_silver = (
    df_silver.withColumn(
        "Phone1",
        concat_ws(
            "-",
            when(col("C_CTRY_1") != "", col("C_CTRY_1")),
            when(col("C_AREA_1") != "", col("C_AREA_1")),
            when(col("C_LOCAL_1") != "", col("C_LOCAL_1")),
            when(col("C_EXT_1") != "", col("C_EXT_1")),
        ),
    )
    .withColumn(
        "Phone2",
        concat_ws(
            "-",
            when(col("C_CTRY_2") != "", col("C_CTRY_2")),
            when(col("C_AREA_2") != "", col("C_AREA_2")),
            when(col("C_LOCAL_2") != "", col("C_LOCAL_2")),
            when(col("C_EXT_2") != "", col("C_EXT_2")),
        ),
    )
    .withColumn(
        "Phone3",
        concat_ws(
            "-",
            when(col("C_CTRY_3") != "", col("C_CTRY_3")),
            when(col("C_AREA_3") != "", col("C_AREA_3")),
            when(col("C_LOCAL_3") != "", col("C_LOCAL_3")),
            when(col("C_EXT_3") != "", col("C_EXT_3")),
        ),
    )
)

#adding timestamp
df_silver=df_silver.withColumn("_load_ts",current_timestamp())

In [0]:
silver_cols=["ActionTS","C_ID","C_TAX_ID","C_GNDR","C_TIER","C_DOB", 
    "C_L_NAME","C_F_NAME","C_M_NAME","C_ADLINE1","C_ADLINE2", 
    "C_ZIPCODE","C_CITY","C_STATE_PROV","C_CTRY","C_PRIM_EMAIL", 
    "C_ALT_EMAIL","C_LCL_TX_ID","C_NAT_TX_ID","_batch","_run_id", 
    "Phone1","Phone2","Phone3","Status","_load_ts"]
df_silver_final=df_silver.select(*silver_cols)
df_silver_final.limit(10).display()

In [0]:
try:
    #appending to silver customer table
    print("Appending start....")
    df_silver_final.write.mode("append").format("delta").saveAsTable(silver_customer)
    print("Append successfully.....")

    run_id=df_silver_final.select("_run_id").first()[0]
    silver_history = spark.sql(f"DESCRIBE HISTORY {silver_customer}").first()
    metrics = silver_history["operationMetrics"]

    source_count = int(metrics.get("numSourceRows", 0))

    inserted = int(metrics.get("numTargetRowsInserted", 0))
    updated = int(metrics.get("numTargetRowsUpdated", 0))
    deleted = int(metrics.get("numTargetRowsDeleted", 0))
    rows_affected = inserted + updated + deleted

    target_count = spark.read.table(silver_customer).count()

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="customer",
        source_layer="bronze",   
        target_layer="silver",     
        source_count=source_count,
        target_count=target_count
    )
    
    # 6. Log Audit Event
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id,
        layer="silver",
        table_name="customer",
        operation="MERGE",          
        rows_affected=rows_affected
    )

except Exception as e:
    print("Failed to append")
    raise e

In [0]:
# spark.table(silver_customer).count()